In [1]:
import json
import urllib.request
import time
from pathlib import Path
from IPython.display import display, Markdown

# =========================================================
# CONFIG
# =========================================================
BASE_DIR     = Path(".")
TEXTS_DIR    = BASE_DIR / "data_texts"
FR_PATH      = TEXTS_DIR / "20000lieues_fr.txt"
JSON_PATH    = BASE_DIR / "Points-escales-chapitres.json"
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

TEST_ESCALES = ["Île de Crespo", "Archipel des Pomotou", "Méditerranée"]

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
fr_text_full = fr_text_full.replace("\r\n", "\n").replace("\r", "\n")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"✅ {len(fr_text_full):,} caractères chargés")
print(f"✅ {len(data)} escales dans le JSON")

# =========================================================
# EXTRACTION DU CHAPITRE
# =========================================================
ROMAINS_FR = {
    1:"PREMIER",2:"II",3:"III",4:"IV",5:"V",6:"VI",7:"VII",
    8:"VIII",9:"IX",10:"X",11:"XI",12:"XII",13:"XIII",14:"XIV",
    15:"XV",16:"XVI",17:"XVII",18:"XVIII",19:"XIX",20:"XX",
    21:"XXI",22:"XXII",23:"XXIII",24:"XXIV"
}

def extraire_chapitre(texte, partie, numero):
    romain = ROMAINS_FR.get(numero, str(numero))
    marqueur = f"CHAPITRE {romain}"
    split = texte.split("FIN DE LA PREMIÈRE PARTIE")
    zone = split[0] if partie == 1 else (split[1] if len(split) > 1 else texte)
    idx = zone.find(marqueur)
    if idx == -1:
        return None
    idx_fin = zone.find("CHAPITRE", idx + len(marqueur))
    return zone[idx:idx_fin].strip() if idx_fin != -1 else zone[idx:].strip()

# =========================================================
# OLLAMA
# =========================================================
def appeler_ollama(prompt):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": 1500}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            return json.loads(resp.read().decode("utf-8")).get("response", "").strip()
    except Exception as e:
        return f"Erreur : {e}"

def prompt_especes(nom_escale, texte):
    return f"""Tu es un naturaliste du XIXe siècle expert de Jules Verne.

Voici un extrait de "Vingt mille lieues sous les mers" — escale : {nom_escale}

TEXTE :
{texte[:10000]}

TÂCHE : Relève toutes les espèces maritimes citées dans ce texte.

Les espèces maritimes comprennent :
- Poissons (bars, sardines, requins, thons, lamproies...)
- Mammifères marins (dauphins, baleines, cachalots, phoques...)
- Crustacés (crabes, crevettes, homards...)
- Mollusques (moules, huîtres, poulpes, pieuvres, nautiles...)
- Coraux et madrépores
- Éponges marines
- Algues et végétaux marins (varechs, zostères, cladostèphes, caulerpes...)
- Échinodermes (étoiles de mer, oursins...)
- Méduses et cnidaires
- Reptiles marins (tortues de mer, serpents de mer...)

RÈGLES STRICTES :
1. Copie EXACTEMENT le nom tel qu'il est écrit par Jules Verne
2. Ajoute le nom latin scientifique entre parenthèses si tu le connais
3. La citation doit être le passage EXACT du texte où le nom apparaît
4. Si vraiment aucune espèce maritime n'est citée, écris : "Aucune espèce maritime citée dans ce passage."
5. Réponse en français uniquement

FORMAT :
━━ FAUNE MARITIME ━━
• [nom Jules Verne] ([nom latin si connu])
  Jules Verne écrit : "[citation exacte courte contenant le nom de l'espèce]"

━━ FLORE MARITIME ━━
• [nom Jules Verne] ([nom latin si connu])
  Jules Verne écrit : "[citation exacte courte contenant le nom de l'espèce]"

N'affiche que les sections qui ont des espèces.
"""

# =========================================================
# TEST SUR 3 ESCALES
# =========================================================
for escale in data:
    nom_fr = escale.get("Escales du Nautilus", {}).get("fr", "")
    if nom_fr not in TEST_ESCALES:
        continue
    ch_info = escale.get("chapitre")
    if not ch_info:
        continue

    partie = ch_info["partie"]
    numero = ch_info["chapitre"]
    ch_fr = extraire_chapitre(fr_text_full, partie, numero)

    print(f"\n🧭 {nom_fr} — Partie {partie}, Ch.{numero} ({len(ch_fr) if ch_fr else 0} car.)")
    print("🤖 Envoi à Ollama...")

    reponse = appeler_ollama(prompt_especes(nom_fr, ch_fr)) if ch_fr else "Chapitre introuvable"

    display(Markdown(f"### 🧭 {nom_fr}\n{reponse}\n---"))
    time.sleep(2)

print("\n✅ Test terminé !")


✅ 908,396 caractères chargés
✅ 32 escales dans le JSON

🧭 Île de Crespo — Partie 1, Ch.15 (16671 car.)
🤖 Envoi à Ollama...


### 🧭 Île de Crespo
━━ FAUNE MARITIME ━━
• Poissons (bars, sardines, requins, thons, lamproies...)
  Jules Verne écrit : "Poissons tels que les bars, les sardines, les requins, les thons, les lamproies..."

• Dauphins
  Jules Verne écrit : "Dauphins"

• Baleines
  Jules Verne écrit : "Baleines"

• Cachalots
  Jules Verne écrit : "Cachalots"

• Phoques
  Jules Verne écrit : "Phoques"

• Crabes
  Jules Verne écrit : "Crabes"

• Crevettes
  Jules Verne écrit : "Crevettes"

• Homards
  Jules Verne écrit : "Homards"

• Poulpes
  Jules Verne écrit : "Poulpes"

• Pieuvres
  Jules Verne écrit : "Pieuvres"

• Nautiles
  Jules Verne écrit : "Nautiles"

• Étoiles de mer
  Jules Verne écrit : "Étoiles de mer"

• Oursins
  Jules Verne écrit : "Oursins"

• Méduses et cnidaires
  Jules Verne écrit : "Méduses et cnidaires"

━━ FLORE MARITIME ━━
• Varechs
  Jules Verne écrit : "Varechs"

• Zostères
  Jules Verne écrit : "Zostères"

• Cladostèphes
  Jules Verne écrit : "Cladostèphes"

• Caulerpes
  Jules Verne écrit : "Caulerpes"

• Porphyria laureolata
  Jules Verne écrit : "Algues très-apéritives, telles que la _Porphyria laureolata_"
---


🧭 Archipel des Pomotou — Partie 1, Ch.19 (21056 car.)
🤖 Envoi à Ollama...


### 🧭 Archipel des Pomotou
━━ FAUNE MARITIME ━━
• Maquereaux (Scomber scombrus)
  Jules Verne écrit : "Ces mollusques appartenaient à l'_ostrea lamellosa_, qui est très-commune en Corse."

• Albicores (Thunnus alalunga)
  Jules Verne écrit : "nous en mangeâmes immodérément, après les avoir ouvertes sur notre table même, suivant le précepte de Sénèque. Ces mollusques appartenaient à l'_ostrea lamellosa_, qui est très-commune en Corse."

• Munérophis (spécimen non identifié)
  Jules Verne écrit : "des variétés d'un serpent de mer nommé munérophis."

• Requins (spécimen non identifié)
  Jules Verne écrit : "nous avions à combattre des requins."

• Dauphins (Tursiops truncatus)
  Jules Verne écrit : "Nous avons vu des dauphins."

• Baleines (Balaenoptera physalus)
  Jules Verne écrit : "nous avions à combattre des baleines."

• Cachalots (Physeter macrocephalus)
  Jules Verne écrit : "nous avons à combattre des cachalots."

• Phoques (Phoca vitulina)
  Jules Verne écrit : "nous avions à combattre des phoques."

• Tortues de mer (Chelonia mydas)
  Jules Verne écrit : "Les tortues de mer massacrèrent les matelots de l'_Union_ et le capitaine Bureau, de Nantes, commandant l'_Aimable-Joséphine'."

• Serpents de mer (spécimen non identifié)
  Jules Verne écrit : "serpent de mer nommé munérophis."

━━ FLORE MARITIME ━━
• Huîtres (Ostrea lamellosa)
  Jules Verne écrit : "Ces mollusques appartenaient à l'_ostrea lamellosa_, qui est très-commune en Corse."
---


🧭 Méditerranée — Partie 2, Ch.7 (22108 car.)
🤖 Envoi à Ollama...


### 🧭 Méditerranée
━━ FAUNE MARITIME ━━
• Lamproie (Lampetra)
  Jules Verne écrit : "des lamproies longues d'un mètre"
• Oxyrhynque (Raja clavata)
  Jules Verne écrit : "des oxyrhinques, sortes de raies"
• Raie (Raia sp.)
  Jules Verne écrit : "autres raies"
• Requin (Squalus sp.)
  Jules Verne écrit : "requins"
• Thon (Thunnus thynnus)
  Jules Verne écrit : "des thons"
• Cachalot (Physeter macrocephalus)
  Jules Verne écrit : "deux ou trois cachalots"
• Dauphin (Globicephala macrorhynchus)
  Jules Verne écrit : "quelques dauphins du genre des globicéphales"
• Phoque (Monachus monachus)
  Jules Verne écrit : "une douzaine de phoques au ventre blanc, au pelage noir"
• Tortue de mer (Caretta caretta)
  Jules Verne écrit : "Conseil croit avoir aperçu une tortue large de six pieds"
• Cacouanne (Eretmochelys imbricata)
  Jules Verne écrit : "quelques cacouannes à carapace allongée"

━━ FAUNE MARITIME - Mammifères marins supplémentaires ━━
• Baleine (Balena sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Hippocampe (Hippocampus hippocampus)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Jouan (Zeuglodon sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Centrise (Centriscus scutatus)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Blennie (Blennius sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Éperlan (Aulacomnion sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Exocet (Exocoetus sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Anchois (Engraulis encrasicolus)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Pagel (Pagellus sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Bogue (Boops boops)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Orphé (Orphinus sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Tétrodon (Tetraodontidae sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Miralet (Sparidae sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Baliste (Balistidae sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Labre (Labridae sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"
• Surmulet (Synodontidae sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"

━━ FAUNE MARITIME - Reptiles marins supplémentaires ━━
• Serpent de mer (Hydrophis sp.)
  Jules Verne écrit : "(aucune espèce maritime citée dans ce passage)"

━━ FLORE MARITIME ━━
• Galéolaire (Galaxea sp.)
  Jules Verne écrit : "une admirable galéolaire orangée"
---


✅ Test terminé !
